# Pulmonary fibrosis simulation — interactive runner

Runs the coupled model from
[`Danpc11/lung-nematic`](https://github.com/Danpc11/lung-nematic): alveolar
architecture, the AT2 → KRT8+ → AT1 epithelial state machine, surfactant-driven
collapse, breathing with strain redistribution, and a confined mesenchyme that
fills collapsed alveoli.

Run the cells in order. Every parameter is a form field — no code editing needed.

**Before trusting any number**, read the caveats in
[`simulations/alveolar/README.md`](https://github.com/Danpc11/lung-nematic/blob/main/simulations/alveolar/README.md).
Two results in this model are negative, and the defect counts are close to a
counting-noise floor.

In [ ]:
#@title 1 · Setup — clone the repository and install dependencies { display-mode: "form" }
import os, subprocess, sys

REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ROOT = "/content/lung-nematic"

if os.path.isdir(ROOT):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "imageio", "imageio-ffmpeg"], check=True)

# Colab keeps imported modules in memory, so a fresh pull is invisible until
# the cached copies are dropped. Purge before importing.
for name in [n for n in list(sys.modules) if n.startswith("simulations")]:
    del sys.modules[name]
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from simulations.alveolar import AlveolarConfig, run_and_record_coupled
import numpy as np, pandas as pd, matplotlib.pyplot as plt

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"repository ready at commit {commit}")
print(f"{len(AlveolarConfig.__dataclass_fields__)} parameters available")

## Parameters

Three groups. Defaults reproduce the two-year progressive run in the repository.

In [ ]:
#@title 2 · Scenario — the lesion and how long to run { display-mode: "form" }

#@markdown Total simulated time and how long the AT2→AT1 block lasts.
years_to_simulate = 2.0  #@param {type:"slider", min:0.25, max:5, step:0.25}
injury_duration_months = 3.3  #@param {type:"slider", min:0, max:24, step:0.1}
timestep_hours = 2.0  #@param [1.0, 2.0, 4.0] {type:"raw"}

#@markdown How completely differentiation fails inside the lesion
#@markdown (1.0 = no block, 0.05 = severe), and how hard AT2 are pushed to try.
repair_failure_factor = 0.10  #@param {type:"slider", min:0.02, max:1.0, step:0.01}
injury_activation_boost = 10.0  #@param {type:"slider", min:1, max:30, step:1}
injury_radius_um = 260.0  #@param {type:"slider", min:80, max:500, step:10}

#@markdown Global clock: multiplies matrix and disease kinetics. Lower = slower
#@markdown disease. Cell birth, death and motility keep their own clock.
rate_scale = 0.08  #@param {type:"slider", min:0.01, max:1.0, step:0.01}

#@markdown Domain size. Larger is more alveoli but slower.
domain_um = 1400.0  #@param {type:"slider", min:800, max:2200, step:100}
random_seed = 0  #@param {type:"integer"}

scenario = dict(
    total_time_h=years_to_simulate * 8760.0,
    injury_duration_h=injury_duration_months * 730.0,
    dt_h=timestep_hours,
    repair_failure_factor=repair_failure_factor,
    injury_activation_boost=injury_activation_boost,
    injury_radius_um=injury_radius_um,
    rate_scale=rate_scale,
    width_um=domain_um, height_um=domain_um,
    seed=random_seed,
)
print(f"{years_to_simulate:.2f} years, block lifted at month {injury_duration_months:.1f}")

In [ ]:
#@title 3 · Biology — the two feedback loops, breathing and cell turnover { display-mode: "form" }

#@markdown **Epithelial loop.** How strongly profibrotic signal keeps cells in
#@markdown the KRT8+ state, and how fast aberrant cells are cleared. Low
#@markdown clearance = apoptosis resistance.
stall_promotion_strength = 25.0  #@param {type:"slider", min:0, max:40, step:0.5}
repair_inhibition_strength = 3.0  #@param {type:"slider", min:0, max:10, step:0.5}
aberrant_clearance_rate = 0.0005  #@param {type:"number"}
aberrant_emt_rate = 0.004  #@param {type:"number"}

#@markdown **Matrix loop.** Collagen deposition against turnover. Their ratio
#@markdown sets the point of no return.
deposition_rate_kPa_per_h = 0.16  #@param {type:"number"}
degradation_rate_per_h = 0.004  #@param {type:"number"}
E_act_kPa = 16.0  #@param {type:"slider", min:4, max:40, step:0.5}

#@markdown **Breathing.** Tidal strain is redistributed over whatever can still
#@markdown deform, so stiff and collapsed regions shed their share onto healthy
#@markdown tissue. Cyclic strain also *suppresses* myofibroblast conversion.
tidal_strain = 0.10  #@param {type:"slider", min:0.02, max:0.25, step:0.01}
strain_protection_strength = 6.0  #@param {type:"slider", min:0, max:20, step:0.5}
strain_tgfb_gain = 0.8  #@param {type:"slider", min:0, max:3, step:0.1}
overstrain_threshold = 0.16  #@param {type:"slider", min:0.05, max:0.4, step:0.01}

#@markdown **Mesenchyme.** Cell shape sets both the excluded volume and the
#@markdown resolvable scale; death rates encode apoptosis resistance.
cell_length_um = 50.0  #@param {type:"slider", min:20, max:90, step:5}
cell_width_um = 11.0  #@param {type:"slider", min:5, max:25, step:1}
fibroblast_death_rate = 0.002  #@param {type:"number"}
myofibroblast_death_rate = 0.0004  #@param {type:"number"}
matrix_immobilization = 25.0  #@param {type:"slider", min:0, max:200, step:5}

#@markdown **Collapse mechanics.**
tissue_recoil_Pa = 420.0  #@param {type:"slider", min:200, max:800, step:10}
induration_time_h = 240.0  #@param {type:"number"}

biology = dict(
    stall_promotion_strength=stall_promotion_strength,
    repair_inhibition_strength=repair_inhibition_strength,
    aberrant_clearance_rate=aberrant_clearance_rate,
    aberrant_emt_rate=aberrant_emt_rate,
    deposition_rate_kPa_per_h=deposition_rate_kPa_per_h,
    degradation_rate_per_h=degradation_rate_per_h,
    E_act_kPa=E_act_kPa,
    tidal_strain=tidal_strain,
    strain_protection_strength=strain_protection_strength,
    strain_tgfb_gain=strain_tgfb_gain,
    overstrain_threshold=overstrain_threshold,
    cell_length_um=cell_length_um, cell_width_um=cell_width_um,
    fibroblast_death_rate=fibroblast_death_rate,
    myofibroblast_death_rate=myofibroblast_death_rate,
    matrix_immobilization=matrix_immobilization,
    tissue_recoil_Pa=tissue_recoil_Pa,
    induration_time_h=induration_time_h,
)

# Warn when the director field will be counting-noise dominated.
import numpy as _np
cell_area = _np.pi * 0.25 * cell_length_um * cell_width_um
r_min = _np.sqrt(30 * cell_area / _np.pi)
print(f"cell footprint {cell_area:.0f} um^2 -> reliable director needs a window "
      f"of radius >= {r_min:.0f} um")
if r_min > 28.0:
    print(f"  NOTE: coarse_grain_um defaults to 28 um, below that floor. "
          f"Defect counts will sit near the noise floor - see the README.")

In [ ]:
#@title 4 · Visualisation and output { display-mode: "form" }

frames_per_run = 60  #@param {type:"slider", min:10, max:150, step:5}
frames_per_second = 8  #@param {type:"slider", min:2, max:24, step:1}
resolution_dpi = 110  #@param {type:"slider", min:60, max:200, step:10}

#@markdown Right-hand panel shows where the breath goes. Turning it off roughly
#@markdown halves the rendering time.
show_strain_panel = True  #@param {type:"boolean"}
show_defects = True  #@param {type:"boolean"}
cell_transparency = 0.55  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
stiffness_colormap = "pink_r"  #@param ["pink_r", "inferno", "magma", "YlOrBr", "bone_r"]
strain_colormap = "RdBu_r"  #@param ["RdBu_r", "coolwarm", "seismic", "PuOr"]

export_mp4 = True  #@param {type:"boolean"}
export_gif = True  #@param {type:"boolean"}

visualisation = dict(
    fps=frames_per_second, dpi=resolution_dpi,
    show_strain_panel=show_strain_panel, show_defects=show_defects,
    cell_alpha=cell_transparency,
    stiffness_cmap=stiffness_colormap, strain_cmap=strain_colormap,
    make_mp4=export_mp4, make_gif=export_gif,
)
print(f"{frames_per_run} frames at {frames_per_second} fps -> "
      f"{frames_per_run/frames_per_second:.1f} s of video")

In [ ]:
#@title 5 · Run the simulation
import time
from dataclasses import replace

config = replace(AlveolarConfig(), **scenario, **biology)
config.validate()

frame_every_h = config.total_time_h / frames_per_run
n_steps = int(config.total_time_h / config.dt_h)
seconds_est = n_steps * 0.011 + frames_per_run * (2.3 if show_strain_panel else 1.2)
print(f"{n_steps:,} steps, {frames_per_run} frames  ->  roughly "
      f"{seconds_est/60:.1f} min. Starting...\n")

start = time.time()
def show_progress(n):
    done = n / (frames_per_run + 1)
    bar = "#" * int(done * 30)
    print(f"\r  [{bar:<30}] {done*100:5.1f}%  {time.time()-start:5.0f}s",
          end="", flush=True)

outputs = run_and_record_coupled(
    config, "/content/results", frame_every_h=frame_every_h,
    progress=show_progress, **visualisation,
)
print(f"\n\nfinished in {time.time()-start:.0f} s")

frame = pd.read_csv(outputs["timeseries"])
final = outputs["final"]
print(f"\nAt {final['time_d']/365:.2f} years:")
print(f"  AT1 coverage         {final['frac_AT1']*100:5.1f} %")
print(f"  aberrant basaloid    {final['frac_aberrant']*100:5.1f} %")
print(f"  alveoli indurated    {final['frac_indurated']*100:5.1f} %")
print(f"  mesenchymal cells    {final['n_mesenchymal']:5d} "
      f"({final['n_myofibroblast']} myofibroblast)")
print(f"  peak stiffness       {final['max_stiffness_kPa']:5.1f} kPa")
print(f"  septal thickness     {final['mean_septal_thickness_um']:5.1f} um")
print(f"  strain amplification {final['strain_amplification']:5.2f} x")

In [ ]:
#@title 6 · Watch it
from base64 import b64encode
from IPython.display import HTML, Image, display

if export_mp4 and "mp4" in outputs:
    data = b64encode(open(outputs["mp4"], "rb").read()).decode()
    display(HTML(f'''<video width="960" controls loop>
        <source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'''))
elif export_gif and "gif" in outputs:
    display(Image(filename=outputs["gif"]))
else:
    print("Enable export_mp4 or export_gif in cell 4 to preview.")

In [ ]:
#@title 7 · Time courses { display-mode: "form" }
#@markdown Figure formatted for print: column-width sizing, hairline axes, no
#@markdown top or right spines, and a colour-blind-safe palette. Saved as PDF
#@markdown (vector) and PNG at 600 dpi alongside the results.
figure_width = "double column (183 mm)"  #@param ["single column (89 mm)", "double column (183 mm)"]
save_vector_pdf = True  #@param {type:"boolean"}

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Colour-blind-safe qualitative palette (Wong). Deliberately not the seaborn
# default, which is not safe for deuteranopia in print.
PALETTE = {
    "blue": "#0072B2", "vermillion": "#D55E00", "green": "#009E73",
    "orange": "#E69F00", "sky": "#56B4E9", "pink": "#CC79A7",
    "brown": "#8C6D5B", "grey": "#999999",
}

MILLIMETRE = 1 / 25.4
width_mm = 89.0 if figure_width.startswith("single") else 183.0
width_in = width_mm * MILLIMETRE

sns.set_theme(style="ticks", context="paper")
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "legend.frameon": False,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "lines.linewidth": 1.1,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,          # embed as editable text, not outlines
    "ps.fonttype": 42,
})

months = frame["time_d"] / 30.4
block_lifted = config.injury_duration_h / 730.0
single = figure_width.startswith("single")
figure, axes = plt.subplots(
    2, 2, figsize=(width_in, width_in * (0.95 if single else 0.55)), sharex=True
)

def finish(ax, ylabel, letter, legend=True):
    sns.despine(ax=ax, trim=False)
    ax.set_ylabel(ylabel)
    ax.axvline(block_lifted, color="0.6", lw=0.5, ls=(0, (2, 2)), zorder=0)
    ax.text(-0.20, 1.04, letter, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="bottom", ha="left")
    if legend:
        ax.legend(loc="best", handlelength=1.2, borderpad=0.2, labelspacing=0.25)

ax = axes[0, 0]
ax.plot(months, frame["frac_AT1"] * 100, color=PALETTE["green"], label="AT1")
ax.plot(months, frame["frac_aberrant"] * 100, color=PALETTE["vermillion"],
        label="aberrant basaloid")
ax.plot(months, frame["frac_KRT8"] * 100, color=PALETTE["orange"], label="KRT8$^+$")
ax.plot(months, frame["frac_denuded"] * 100, color=PALETTE["grey"], label="denuded")
finish(ax, "epithelium (%)", "a")

ax = axes[0, 1]
ax.plot(months, frame["frac_open"] * 100, color=PALETTE["blue"], label="open")
ax.plot(months, frame["frac_collapsed"] * 100, color=PALETTE["sky"], label="collapsed")
ax.plot(months, frame["frac_indurated"] * 100, color=PALETTE["brown"], label="indurated")
finish(ax, "alveoli (%)", "b")

ax = axes[1, 0]
ax.plot(months, frame["n_mesenchymal"], color=PALETTE["blue"], label="mesenchymal")
ax.plot(months, frame["n_myofibroblast"], color=PALETTE["vermillion"],
        label="myofibroblast")
finish(ax, "cells (n)", "c")
ax.set_xlabel("time (months)")

ax = axes[1, 1]
ax.plot(months, frame["max_stiffness_kPa"], color=PALETTE["brown"],
        label="peak stiffness (kPa)")
ax.plot(months, frame["mean_septal_thickness_um"], color=PALETTE["pink"],
        label="septal thickness (µm)")
twin = ax.twinx()
twin.plot(months, frame["strain_amplification"], color=PALETTE["sky"],
          ls=(0, (3, 1.5)), label="strain amplification")
twin.set_ylabel("strain / normal")
twin.spines["top"].set_visible(False)
twin.spines["right"].set_linewidth(0.5)
finish(ax, "kPa  |  µm", "d", legend=False)
ax.set_xlabel("time (months)")
handles = ax.get_lines() + twin.get_lines()
ax.legend(handles, [h.get_label() for h in handles], loc="upper left",
          handlelength=1.2, borderpad=0.2, labelspacing=0.25)

figure.align_ylabels(axes)
figure.tight_layout(pad=0.4, w_pad=1.6, h_pad=0.8)

from pathlib import Path
results_dir = Path("/content/results")
figure.savefig(results_dir / "time_courses.png", dpi=600)
if save_vector_pdf:
    figure.savefig(results_dir / "time_courses.pdf")
plt.show()

print(f"figure saved at {width_mm:.0f} mm wide "
      f"({'PDF + PNG 600 dpi' if save_vector_pdf else 'PNG 600 dpi'})")
print("dashed vertical line marks the lifting of the AT2→AT1 block")

In [ ]:
#@title 8 · Diagnostics — are these numbers measurements? { display-mode: "form" }
#@markdown Runs the two checks that decide whether the run can be quoted: the
#@markdown counting-noise floor for the director field, and whether the focus
#@markdown actually localised to collapsed alveoli.
import numpy as np
from scipy.spatial import cKDTree
from simulations.alveolar.model import OPEN

sim = outputs["simulation"]
mes, epi = sim.mesenchyme, sim.epithelium

# --- 1. counting-noise floor -------------------------------------------------
# For N randomly oriented objects, noise alone gives an apparent order ~1/sqrt(N),
# so a window needs roughly 30 samples before order means anything.
cell_area = config.cell_area_um2
r_min = np.sqrt(30 * cell_area / np.pi)

points = np.column_stack([mes.x, mes.y])
tree = cKDTree(points)
neighbours = np.array([
    len(tree.query_ball_point(p, config.coarse_grain_um)) for p in points
])
field = mes.director_field()
packing = field["density"] * cell_area / mes.grid_step ** 2
gated = packing >= config.min_packing_for_nematic
measured_S = float(np.median(field["order"][gated])) if gated.any() else float("nan")
n_per_window = float(np.median(neighbours)) if len(neighbours) else 0.0
noise_floor = 1.0 / np.sqrt(max(n_per_window, 1.0))

print("COUNTING-NOISE FLOOR")
print(f"  cell footprint            {cell_area:6.0f} um^2")
print(f"  window needed (R_min)     {r_min:6.0f} um   = sqrt(30 * A / pi)")
print(f"  window in use             {config.coarse_grain_um:6.0f} um")
print(f"  cells per window (median) {n_per_window:6.0f}")
print(f"  noise floor  |S| ~        {noise_floor:6.2f}")
print(f"  measured     |S| =        {measured_S:6.2f}")
if measured_S > noise_floor:
    print("  -> above the floor: defect counts can be interpreted")
else:
    print("  -> NOISE-LIMITED: the defect counts in the video are detector")
    print("     artefacts, not measurements. Do not quote them. Either widen")
    print("     coarse_grain_um past R_min, or restrict the analysis to filled")
    print("     collapsed alveoli where 2D density is genuinely high.")

# --- 2. did the focus form where the hypothesis says? -------------------------
label = mes.alveolus_label
ix = np.clip((mes.x / mes.grid_step).astype(int), 0, mes.nx - 1)
iy = np.clip((mes.y / mes.grid_step).astype(int), 0, mes.ny - 1)
owner = label[iy, ix]
inside = (owner >= 0) & (epi.alveolar_state[np.clip(owner, 0, None)] != OPEN)

valid = label >= 0
collapsed_area = float((epi.alveolar_state[label[valid]] != OPEN).mean())
fraction_inside = float(inside.mean()) if mes.n_cells else 0.0

print()
print("FOCUS LOCALISATION")
print(f"  mesenchymal cells inside collapsed alveoli  {int(inside.sum()):5d} / {mes.n_cells}"
      f"  ({fraction_inside*100:.0f} %)")
print(f"  collapsed tissue as a share of area          {collapsed_area*100:.0f} %")
if collapsed_area > 0:
    print(f"  enrichment over chance                       {fraction_inside/collapsed_area:.2f} x")
    print("  (>1 means collapse concentrates the mesenchyme, which is the claim)")
print()
print("One realisation, one seed. Repeat across seeds before quoting an effect size.")

In [ ]:
#@title 9 · Retrospective drug controls { display-mode: "form" }
#@markdown Runs known interventions through the *reduced* coupled model (fast)
#@markdown and scores them against the clinical record. Agreement with drugs
#@markdown that worked proves little; the discriminating test is whether the
#@markdown model reproduces the ones that **failed** - LOXL2/simtuzumab and
#@markdown single-target alpha-v-beta-6 blockade.
dosing = "treatment (established disease)"  #@param ["prevention (from day 0)", "treatment (established disease)"]
use_matrix_maturation = True  #@param {type:"boolean"}

from simulations.coupled_analysis import CoupledParameters
from simulations.pharmacology import run_panel, run_panel_with_maturation, score

reduced = CoupledParameters.from_alveolar(config)
start_h = 0.0 if dosing.startswith("prevention") else 13140.0

if use_matrix_maturation:
    panel = run_panel_with_maturation(reduced, treatment_start_h=start_h)
else:
    panel = run_panel(reduced)

print(f"{'intervention':26s} {'E final':>8s}  {'model':12s} {'clinical record':22s}")
print("-" * 76)
for _, row in panel.iterrows():
    if row["expected"] == "-":
        print(f"{row['intervention']:26s} {row['E_final_kPa']:8.2f}  (untreated reference)")
        continue
    mark = "ok" if row["agrees"] else "MISMATCH"
    print(f"{row['intervention']:26s} {row['E_final_kPa']:8.2f}  "
          f"{row['response']:12s} {row['expected']:22s} {mark}")

result = score(panel)
print()
print(f"clinical failures reproduced: {result['clinical_failures_reproduced']}"
      f"/{result['clinical_failures_total']}")
print(f"passes the discriminating test: {result['passes_discriminating_test']}")
if not result["passes_discriminating_test"]:
    print()
    print("A model that cannot reproduce a clinical failure is fitted to")
    print("biological plausibility. Treat its target rankings accordingly.")

In [ ]:
#@title 10 · Export everything { display-mode: "form" }
import json, shutil, zipfile
from dataclasses import asdict
from pathlib import Path

bundle_name = "ipf_simulation_run"  #@param {type:"string"}
also_include_frames = False  #@param {type:"boolean"}

results = Path("/content/results")
bundle = Path(f"/content/{bundle_name}.zip")

with open(results / "config.json", "w") as handle:
    json.dump(asdict(config), handle, indent=2)
frame.to_csv(results / "timeseries.csv", index=False)

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in results.iterdir():
        if item.is_file():
            archive.write(item, item.name)
    if also_include_frames and (results / "frames").is_dir():
        for item in sorted((results / "frames").glob("*.png")):
            archive.write(item, f"frames/{item.name}")

size_mb = bundle.stat().st_size / 1e6
print(f"{bundle.name}  ({size_mb:.1f} MB)")
print("contains: video, time series, and the config that reproduces this run")

try:
    from google.colab import files
    files.download(str(bundle))
except Exception as error:
    print(f"\nAutomatic download unavailable ({error}).")
    print(f"Use the file browser on the left: {bundle}")

## Reproducing a run later

The exported `config.json` holds every parameter. To rerun it:

```python
import json
from dataclasses import replace
from simulations.alveolar import AlveolarConfig, run_and_record_coupled

with open("config.json") as handle:
    config = replace(AlveolarConfig(), **json.load(handle))
run_and_record_coupled(config, "results/rerun", frame_every_h=292.0)
```

Parameter sets backing a figure belong in `simulations/configs/`, which is
version-controlled — unlike the results directory.

## Reading the output honestly

**Cell 8 decides whether the defect counts mean anything.** For N randomly
oriented objects, counting noise alone produces an apparent order of about
`1/sqrt(N)`, so a smoothing window needs roughly 30 cells before measured order
can be distinguished from chance. With the default 50×11 µm cell that is a
64 µm radius, against a `coarse_grain_um` of 28. When the cell reports
NOISE-LIMITED, the `+n/-n` markers in the video are detector artefacts and must
not be quoted. Architecture, collapse, septal thickening, focus localisation and
strain redistribution are unaffected — those are measurements.

**Cell 9 is the check that keeps the model honest.** Agreement with targets that
worked proves nothing; every implicated target is load-bearing inside a
mechanistic model by construction. The discriminating test is whether the model
reproduces interventions that *failed*: LOXL2/simtuzumab, terminated for
futility, and single-target αvβ6 blockade, whose trials were terminated or
discontinued. The model currently reproduces one of those two, so it does not
pass. That is reported rather than hidden.

**The first weeks are a transient.** `n_resident_fibroblasts` seeds 260 cells,
above the homeostatic density of the confined interstitium, so the population
falls before the lesion drives it up — recovered by about month 4 in a two-year
run. Discard the opening months or lower the seed.

**Every result here is one realisation with one seed.** Repeat across seeds
before quoting an effect size.